In [1]:
import polars as pl
import numpy as np
from scipy import stats
from pathlib import Path

In [2]:
DONOR = 'donor_4'

In [3]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined' / DONOR

In [4]:
# One sample test
def one_sample_test_against_chance(accuracies, p_null=0.5):
    accuracies = np.asarray(accuracies, dtype=float)
    n = len(accuracies)
    mean = accuracies.mean()
    sd = accuracies.std(ddof=1)
    se = sd / np.sqrt(n)
 
    # one-sample t-test -- preferred for small n (e.g. n=5 runs)
    t_stat, t_p = stats.ttest_1samp(accuracies, popmean=p_null)
 
    # z-test shown for reference; not appropriate for n this small
    z_stat = (mean - p_null) / se
    z_p = 2 * (1 - stats.norm.cdf(abs(z_stat)))
 
    return {
        "n_runs": n,
        "mean_accuracy": mean,
        "sd": sd,
        "t_stat": t_stat,
        "t_p_value": t_p,
        "z_stat": z_stat,
        "z_p_value": z_p,
    }

In [5]:
# Pretty report
def report(name, accuracies):
    r = one_sample_test_against_chance(accuracies)
    print(f"{name}:")
    print(f"  mean = {r['mean_accuracy']*100:.2f}% +/- {r['sd']*100:.2f}% "
          f"(n={r['n_runs']} runs)")
    print(f"  t-test:  t({r['n_runs']-1}) = {r['t_stat']:.3f}, "
          f"p = {r['t_p_value']:.4g}")
    print(f"  z-test:  z = {r['z_stat']:.3f}, p = {r['z_p_value']:.4g}")
    print()

In [6]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [7]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",39,71.0,0.78,71.5,0.8058,0.6984,0.761,0.7283,0.888
"""LogisticRegression""",1011,"""H3K4me3""",null,69.8,0.7644,71.8,0.7992,0.7523,0.6534,0.6994,0.762
"""RandomForest""",1011,"""H3K4me3""",null,70.4,0.7813,72.1,0.8057,0.7528,0.6614,0.7041,0.767
"""SVM_Linear""",1011,"""H3K4me3""",null,69.9,0.7636,71.7,0.801,0.7506,0.6534,0.6986,0.76
"""DirectRanker""",123,"""H3K4me3""",60,74.1,0.83,71.6,0.8145,0.6891,0.7703,0.7274,0.892
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,73.9,0.8033,71.4,0.755,0.7315,0.6502,0.6885,0.741
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",80,72.4,0.81,74.9,0.8361,0.7386,0.7753,0.7565,0.927
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,69.7,0.7381,71.1,0.7686,0.7477,0.6421,0.6909,0.735


# Chek for the H3K9me3 results

In [8]:
# Check for the H3K9me3
H3K9me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K9me3"))
H3K9me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K9me3""",100,49.4,0.51,50.6,0.5152,0.5042,0.9641,0.6621,0.09
"""DirectRanker""",123,"""H3K9me3""",100,50.9,0.52,51.0,0.5371,0.501,0.9715,0.6611,0.091
"""DirectRanker""",42,"""H3K9me3""",100,48.4,0.53,51.3,0.5501,0.4968,0.9771,0.6587,0.112
"""DirectRanker""",456,"""H3K9me3""",100,49.4,0.53,51.9,0.5402,0.5026,0.9794,0.6643,0.097
"""DirectRanker""",789,"""H3K9me3""",100,49.6,0.53,52.3,0.5332,0.5136,0.9742,0.6726,0.091


In [9]:
# Select the accuracy
test_accuracies = H3K9me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.506, 0.51 , 0.513, 0.519, 0.523])

In [10]:
report("H3K9me3", test_accuracies)

H3K9me3:
  mean = 51.42% +/- 0.68% (n=5 runs)
  t-test:  t(4) = 4.646, p = 0.009688
  z-test:  z = 4.646, p = 3.378e-06



# Check for H3K27me3

In [11]:
# Check for the H3K27me3
H3K27me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K27me3"))
H3K27me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K27me3""",100,50.2,0.53,51.1,0.5157,0.5066,0.994,0.6711,0.038
"""DirectRanker""",123,"""H3K27me3""",100,51.9,0.53,50.4,0.5255,0.498,0.9898,0.6626,0.046
"""DirectRanker""",42,"""H3K27me3""",100,48.3,0.52,49.6,0.5206,0.4883,0.9938,0.6548,0.039
"""DirectRanker""",456,"""H3K27me3""",100,48.8,0.53,49.2,0.5159,0.4888,0.9877,0.654,0.038
"""DirectRanker""",789,"""H3K27me3""",100,48.4,0.52,51.6,0.5186,0.5098,0.9861,0.6721,0.049


In [12]:
# Select the accuracy
test_accuracies = H3K27me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.511, 0.504, 0.496, 0.492, 0.516])

In [13]:
report("H3K27me3", test_accuracies)

H3K27me3:
  mean = 50.38% +/- 1.00% (n=5 runs)
  t-test:  t(4) = 0.849, p = 0.4438
  z-test:  z = 0.849, p = 0.396

